In [ ]:
from keras.models import Sequential
from keras.layers import Activation, Dense, Dropout, LSTM, Bidirectional, GRU, SimpleRNN
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn import preprocessing
import numpy as np
import pandas as pd
from ta import add_all_ta_features # Library that does financial technical analysis 

from sklearn.preprocessing import MinMaxScaler 
scaler = MinMaxScaler(feature_range=(0, 1))

import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

In [ ]:
hist = pd.read_csv('Data/WAFA-ASSURANCE.csv', index_col="Date", parse_dates=True)

# Add all technical analysis to the dataframe we've already loaded
hist = add_all_ta_features(hist, "Open", "High", "Low", "Close", "Volume", fillna=True) 

target_col = 'Close'

window_len = 20
test_size = 0.2
zero_base = True
print(hist)

In [ ]:
def train_test_split(df, test_size=0.2):
    split_row = len(df) - int(test_size * len(df))
    train_data = df.iloc[:split_row]
    test_data = df.iloc[split_row:]
    return train_data, test_data

train, test = train_test_split(hist, test_size=0.2)

In [ ]:
def line_plot(line1, line2, label1=None, label2=None, title='Close Price history', lw=2):
    fig, ax = plt.subplots(1, figsize=(16, 8))
    ax.plot(line1, label=label1, linewidth=lw)
    ax.plot(line2, label=label2, linewidth=lw)
    ax.set_ylabel('Price', fontsize=14)
    ax.set_title(title, fontsize=16)
    ax.legend(loc='best', fontsize=16)
    plt.show()

line_plot(train[target_col], test[target_col], 'training', 'test', title='Close Price history')

In [ ]:
def extract_window_data(df, window_len=5, zero_base=True):
    window_data = []
    for idx in range(len(df) - window_len):
        tmp = df[idx: (idx + window_len)].copy()
        if zero_base:
            X = tmp.values
            # X = preprocessing.scale(X)
            X = scaler.fit_transform(X)
        window_data.append(X)
    return np.array(window_data)

In [ ]:
def prepare_data(df, target_col, window_len=5, zero_base=True, test_size=0.2):
    train_data, test_data = train_test_split(df, test_size=test_size)
    X_train = extract_window_data(train_data, window_len, zero_base)
    X_test = extract_window_data(test_data, window_len, zero_base)
    y_train = train_data[target_col][window_len:].values
    y_test = test_data[target_col][window_len:].values
    if zero_base:
        y_train = y_train / train_data[target_col][:-window_len].values - 1
        y_test = y_test / test_data[target_col][:-window_len].values - 1
    return train_data, test_data, X_train, X_test, y_train, y_test

In [ ]:
train, test, X_train, X_test, y_train, y_test = prepare_data(
    hist, target_col, window_len=window_len, zero_base=zero_base, test_size=test_size)

targets = test[target_col][window_len:]

In [ ]:
def build_lstm_model(X_train, y_train, X_test, y_test):
    # The LSTM architecture
    regressorLSTM = Sequential()    
    regressorLSTM.add(LSTM(256, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
    regressorLSTM.add(Dropout(0.2))
    regressorLSTM.add(LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
    regressorLSTM.add(Dropout(0.2))
    regressorLSTM.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
    regressorLSTM.add(Dropout(0.2))
    regressorLSTM.add(Dense(32,kernel_initializer="uniform",activation='relu'))        
    regressorLSTM.add(Dense(1,kernel_initializer="uniform",activation='linear'))
    regressorLSTM.add(Activation("linear"))

    regressorLSTM.compile(loss="mean_squared_error", optimizer="adam", metrics=['accuracy'])
    regressorLSTM.fit(X_train, y_train, epochs=100, batch_size=96)
    lstm_pred = regressorLSTM.predict(X_test).squeeze()

    lstm_MSE = mean_squared_error(y_test, lstm_pred)
    lstm_MAE = mean_absolute_error(y_test, lstm_pred)
    lstm_RMSE = np.sqrt(np.mean(np.power((np.array(y_test)-np.array(lstm_pred)),2)))
    print('LSTM Mean Absolute Error: {}'.format(lstm_MAE))
    print('LSTM MSE: {}'.format(lstm_MSE))
    print('LSTM RMSE: {}'.format(lstm_RMSE))

    return regressorLSTM, lstm_pred, lstm_MSE, lstm_MAE, lstm_RMSE

In [ ]:
regressorLSTM, lstm_pred, lstm_MSE, lstm_MAE, lstm_RMSE = build_lstm_model(X_train, y_train, X_test, y_test)

In [ ]:
def build_bilstm_model(X_train, y_train, X_test, y_test):
    # The Bidirectional LSTM architecture
    regressorBiLSTM = Sequential()    
    regressorBiLSTM.add(Bidirectional(LSTM(256, input_shape=(X_train.shape[1], X_train.shape[2]), dropout=0.2, return_sequences=True)))
    regressorBiLSTM.add(Bidirectional(LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2]), dropout=0.2, return_sequences=True)))
    regressorBiLSTM.add(Bidirectional(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), dropout=0.2, return_sequences=False)))
    regressorBiLSTM.add(Dense(32,kernel_initializer="uniform",activation='relu'))        
    regressorBiLSTM.add(Dense(1,kernel_initializer="uniform",activation='linear'))
    regressorBiLSTM.add(Activation("linear"))

    regressorBiLSTM.compile(loss="mean_squared_error", optimizer="adam", metrics=['accuracy'])
    regressorBiLSTM.fit(X_train, y_train, epochs=100, batch_size=96)
    bilstm_pred = regressorBiLSTM.predict(X_test).squeeze()

    bilstm_MSE = mean_squared_error(y_test, bilstm_pred)
    bilstm_MAE = mean_absolute_error(y_test, bilstm_pred)
    bilstm_RMSE = np.sqrt(np.mean(np.power((np.array(y_test)-np.array(bilstm_pred)),2)))
    print('Bidirectional LSTM Mean Absolute Error: {}'.format(bilstm_MAE))
    print('Bidirectional LSTM MSE: {}'.format(bilstm_MSE))
    print('Bidirectional LSTM RMSE: {}'.format(bilstm_RMSE))

    return regressorBiLSTM, bilstm_pred, bilstm_MSE, bilstm_MAE, bilstm_RMSE

In [ ]:
regressorBiLSTM, bilstm_pred, bilstm_MSE, bilstm_MAE, bilstm_RMSE = build_bilstm_model(X_train, y_train, X_test, y_test)

Epoch 79/100
34/34 [==============================] - 3s 95ms/step - loss: 3.3172e-04 - accuracy: 0.0462
Epoch 80/100
34/34 [==============================] - 3s 93ms/step - loss: 3.4715e-04 - accuracy: 0.0462
Epoch 81/100
34/34 [==============================] - 3s 96ms/step - loss: 3.4642e-04 - accuracy: 0.0462
Epoch 82/100
34/34 [==============================] - 3s 95ms/step - loss: 3.1692e-04 - accuracy: 0.0462
Epoch 83/100
34/34 [==============================] - 3s 97ms/step - loss: 3.2490e-04 - accuracy: 0.0462
Epoch 84/100
34/34 [==============================] - 3s 93ms/step - loss: 3.2645e-04 - accuracy: 0.0462
Epoch 85/100
34/34 [==============================] - 3s 93ms/step - loss: 2.9024e-04 - accuracy: 0.0462
Epoch 86/100
34/34 [==============================] - 3s 91ms/step - loss: 2.9982e-04 - accuracy: 0.0462
Epoch 87/100
34/34 [==============================] - 3s 95ms/step - loss: 2.9787e-04 - accuracy: 0.0462
Epoch 88/100
34/34 [==============================] - 3

In [ ]:
def build_gru_model(X_train, y_train, X_test, y_test):
    # The GRU architecture
    regressorGRU = Sequential()
    regressorGRU.add(GRU(256, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
    regressorGRU.add(Dropout(0.2))
    regressorGRU.add(GRU(128, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
    regressorGRU.add(Dropout(0.2))
    regressorGRU.add(GRU(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
    regressorGRU.add(Dropout(0.2))
    regressorGRU.add(Dense(32,kernel_initializer="uniform",activation='relu'))        
    regressorGRU.add(Dense(1,kernel_initializer="uniform",activation='linear'))
    regressorGRU.add(Activation("linear"))
    
    regressorGRU.compile(loss="mean_squared_error", optimizer="adam", metrics=['accuracy'])
    regressorGRU.fit(X_train, y_train, epochs=100, batch_size=96)
    gru_pred = regressorGRU.predict(X_test).squeeze()

    gru_MSE = mean_squared_error(y_test, gru_pred)
    gru_MAE = mean_absolute_error(y_test, gru_pred)
    gru_RMSE = np.sqrt(np.mean(np.power((np.array(y_test)-np.array(gru_pred)),2)))
    print('GRU Mean Absolute Error: {}'.format(gru_MAE))
    print('GRU MSE: {}'.format(gru_MSE))
    print('GRU RMSE: {}'.format(gru_RMSE))

    return regressorGRU, gru_pred, gru_MSE, gru_MAE, gru_RMSE

In [ ]:
regressorGRU, gru_pred, gru_MSE, gru_MAE, gru_RMSE = build_gru_model(X_train, y_train, X_test, y_test)

In [ ]:
def build_rnn_model(X_train, y_train, X_test, y_test):
    # The RNN architecture
    regressorRNN = Sequential()
    regressorRNN.add(SimpleRNN(256, return_sequences=True))
    regressorRNN.add(SimpleRNN(128, return_sequences=True))
    regressorRNN.add(SimpleRNN(64, return_sequences=True))
    regressorRNN.add(SimpleRNN(32))
    regressorRNN.add(Dense(1))
    regressorRNN.add(Activation("linear"))

    regressorRNN.compile(loss="mean_squared_error", optimizer="adam", metrics=['accuracy'])
    regressorRNN.fit(X_train, y_train, epochs=100, batch_size=96)
    rnn_pred = regressorRNN.predict(X_test).squeeze()

    rnn_MSE = mean_squared_error(y_test, rnn_pred)
    rnn_MAE = mean_absolute_error(y_test, rnn_pred)
    rnn_RMSE = np.sqrt(np.mean(np.power((np.array(y_test)-np.array(rnn_pred)),2)))
    print('RNN Mean Absolute Error: {}'.format(rnn_MAE))
    print('RNN MSE: {}'.format(rnn_MSE))
    print('RNN RMSE: {}'.format(rnn_RMSE))

    return regressorRNN, rnn_pred, rnn_MSE, rnn_MAE, rnn_RMSE

In [ ]:
regressorRNN, rnn_pred, rnn_MSE, rnn_MAE, rnn_RMSE = build_rnn_model(X_train, y_train, X_test, y_test)

In [ ]:
lstm_pred = test[target_col].values[:-window_len] * (lstm_pred + 1)
lstm_pred = pd.Series(index=targets.index, data=lstm_pred)

bilstm_pred = test[target_col].values[:-window_len] * (bilstm_pred + 1)
bilstm_pred = pd.Series(index=targets.index, data=bilstm_pred)

gru_pred = test[target_col].values[:-window_len] * (gru_pred + 1)
gru_pred = pd.Series(index=targets.index, data=gru_pred)

rnn_pred = test[target_col].values[:-window_len] * (rnn_pred + 1)
rnn_pred = pd.Series(index=targets.index, data=rnn_pred)

In [ ]:
line_plot(targets, lstm_pred, 'actual', 'prediction', lw=3)

In [ ]:
line_plot(targets, bilstm_pred, 'actual', 'prediction', lw=3)

In [ ]:
line_plot(targets, gru_pred, 'actual', 'prediction', lw=3)

In [ ]:
line_plot(targets, rnn_pred, 'actual', 'prediction', lw=3)

In [ ]:
result = pd.concat([test[target_col], lstm_pred, bilstm_pred, gru_pred, rnn_pred], axis=1)
result.rename(columns={result.columns[0]:'Close Test'}, inplace=True)
result.rename(columns={result.columns[1]:'Close LSTM'}, inplace=True)
result.rename(columns={result.columns[2]:'Close BiLSTM'}, inplace=True)
result.rename(columns={result.columns[3]:'Close GRU'}, inplace=True)
result.rename(columns={result.columns[4]:'Close RNN'}, inplace=True)

In [ ]:
result = result.dropna()
result

In [ ]:
#result.to_excel("result_lstm_gru_rnn.xlsx")

In [ ]:
lstm_MAPE = mean_absolute_percentage_error(result["Close Test"], result["Close LSTM"])
bilstm_MAPE = mean_absolute_percentage_error(result["Close Test"], result["Close BiLSTM"])
gru_MAPE = mean_absolute_percentage_error(result["Close Test"], result["Close GRU"])
rnn_MAPE = mean_absolute_percentage_error(result["Close Test"], result["Close RNN"])

In [ ]:
cnames=['LSTM', 'BiLSTM', 'GRU', 'RNN']
finalResult = pd.DataFrame(columns=cnames, index=["MAE", "MSE", "RMSE", "MAPE"])

In [ ]:
finalResult["LSTM"]["MAE"] = lstm_MAE
finalResult["LSTM"]["MSE"] = lstm_MSE
finalResult["LSTM"]["RMSE"] = lstm_RMSE
finalResult["LSTM"]["MAPE"] = lstm_MAPE

finalResult["BiLSTM"]["MAE"] = bilstm_MAE
finalResult["BiLSTM"]["MSE"] = bilstm_MSE
finalResult["BiLSTM"]["RMSE"] = bilstm_RMSE
finalResult["BiLSTM"]["MAPE"] = bilstm_MAPE

finalResult["GRU"]["MAE"] = gru_MAE
finalResult["GRU"]["MSE"] = gru_MSE
finalResult["GRU"]["RMSE"] = gru_RMSE
finalResult["GRU"]["MAPE"] = gru_MAPE

finalResult["RNN"]["MAE"] = rnn_MAE
finalResult["RNN"]["MSE"] = rnn_MSE
finalResult["RNN"]["RMSE"] = rnn_RMSE
finalResult["RNN"]["MAPE"] = rnn_MAPE

In [ ]:
finalResult

In [ ]:
finalResult.to_excel("finalResult_lstm_bilstm_gru_rnn_WAFA-ASSURANCE.xlsx")